<a href="https://colab.research.google.com/github/AditiWagh1/CCD-Miniproject/blob/main/DLMiniproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
pranay22077_dfdc_10_path = kagglehub.dataset_download('pranay22077/dfdc-10')

print('Data source import complete.')


In [ ]:
!pip install flask-cors pyngrok mediapipe librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
!pip install mediapipe

In [ ]:
import pandas as pd
import json
import os
import glob

# This finds ALL metadata files in any subfolder of your input
json_files = glob.glob('/kaggle/input/**/metadata.json', recursive=True)

all_metadata = []
for file_path in json_files:
    with open(file_path, 'r') as f:
        data = json.load(f)
        # Convert dictionary to a list of records including the file path
        for filename, info in data.items():
            info['filename'] = filename
            # Store the actual path to the video file for later use
            info['directory'] = os.path.dirname(file_path)
            all_metadata.append(info)

df = pd.DataFrame(all_metadata)
print(f"Total videos ready for processing: {len(df)}")
print(df[['filename', 'label', 'original']].head())

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import librosa
import numpy as np

# Get the path of the first video in your new combined dataframe
video_name = df.iloc[0]['filename']
video_dir = df.iloc[0]['directory']
video_path = os.path.join(video_dir, video_name)

# Extract audio and convert to Mel-spectrogram
y, sr = librosa.load(video_path, sr=16000) # Standard 16kHz for speech
mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
mel_db = librosa.power_to_db(mel_spec, ref=np.max)

print(f"Audio Token Shape: {mel_db.shape}") # This is your 'Speech Token'

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# This one is always pre-installed on Kaggle
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def get_mouth_crop_proportional(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    for (x, y, w, h) in faces:
        # The mouth is mathematically located in the lower-center of the face box
        # We define the mouth area as:
        # Horizontal: Middle 50% of the face width
        # Vertical: Bottom 35% of the face height

        mx = x + int(w * 0.25)
        my = y + int(h * 0.65)
        mw = int(w * 0.5)
        mh = int(h * 0.25)

        mouth_crop = frame[my:my+mh, mx:mx+mw]

        if mouth_crop.size > 0:
            return cv2.resize(mouth_crop, (112, 112))

    return None

# Test on your video
# test_crop = get_mouth_crop_proportional(your_frame)

In [ ]:
import torch

def create_input_tensors(visual_list, audio_spec):
    # 1. Convert list of images to a single 4D Tensor (Frames, Channels, H, W)
    visual_tensor = torch.from_numpy(np.array(visual_list)).permute(0, 3, 1, 2).float() / 255.0

    # 2. Audio is already a 2D array, convert to Tensor
    audio_tensor = torch.from_numpy(audio_spec).float().unsqueeze(0)

    return visual_tensor, audio_tensor

print("Data alignment function ready!")

Data alignment function ready!


In [ ]:
import numpy as np

def process_video_to_tokens(video_path, max_frames=30):
    cap = cv2.VideoCapture(video_path)
    frames = []

    # Extract 30 frames (roughly 1 second of video)
    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret: break

        mouth = get_mouth_crop_proportional(frame) # Your working function
        if mouth is not None:
            frames.append(mouth)

    cap.release()

    # 1. Prepare Visual Token (Frames, Height, Width, Channels)
    visual_tokens = np.array(frames)

    # 2. Prepare Audio Token (Match the duration of the video frames)
    # We use librosa to get the corresponding 1 second of audio
    y, sr = librosa.load(video_path, sr=16000, duration=1.0)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    audio_tokens = librosa.power_to_db(mel_spec, ref=np.max)

    return visual_tokens, audio_tokens

In [ ]:
import torch
import torch.nn as nn

class CrossModalTransformer(nn.Module):
    def __init__(self):
        super(CrossModalTransformer, self).__init__()
        # Audio Encoder (Simple CNN to process spectrogram)
        self.audio_enc = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # Visual Encoder (CNN to process mouth frames)
        self.visual_enc = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3, 3, 3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d((1, 2, 2))
        )

        # The "Brain": Cross-Attention Layer
        self.multihead_attn = nn.MultiheadAttention(embed_dim=128, num_heads=4)

    def forward(self, audio, visual):
        # We will write the logic for the "Brain" next!
        pass

print("Model shell is ready!")

Model shell is ready!


In [ ]:
def get_aligned_sample(video_path, target_frames=30):
    cap = cv2.VideoCapture(video_path)
    frames = []

    # Extract frames
    while len(frames) < target_frames:
        ret, frame = cap.read()
        if not ret:
            break
        crop = get_mouth_crop_proportional(frame)
        if crop is not None:
            frames.append(crop)
    cap.release()

    if len(frames) == 0:
        return None, None

    # --- THE FIX: PADDING ---
    # If we have fewer than 30 frames, repeat the last frame until we hit 30
    while len(frames) < target_frames:
        frames.append(frames[-1])

    # If we somehow have more (though the loop prevents it), truncate
    frames = frames[:target_frames]

    # 2. Get Audio (Match the 1.0 second duration)
    y, sr = librosa.load(video_path, sr=16000, duration=1.0)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)

    # Ensure audio length is consistent too (e.g., 32 time steps)
    if mel_spec.shape[1] < 32:
        padding = 32 - mel_spec.shape[1]
        mel_spec = np.pad(mel_spec, ((0, 0), (0, padding)), mode='constant')
    else:
        mel_spec = mel_spec[:, :32]

    audio_tokens = librosa.power_to_db(mel_spec, ref=np.max)

    return np.array(frames), audio_tokens
# Test it
# v_sample, a_sample = get_aligned_sample(test_video_path)
# print(f"Visual: {v_sample.shape}, Audio: {a_sample.shape}")

In [ ]:
import torch

# Check if the P100 GPU is available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Current device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Current device: cpu


In [ ]:
class DeepfakeTransformer(nn.Module):
    def __init__(self, embed_dim=128):
        super(DeepfakeTransformer, self).__init__()

        # 1. Visual Feature Extractor (3D CNN for video)
        self.vis_conv = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.vis_proj = nn.Linear(32, embed_dim)

        # 2. Audio Feature Extractor (2D CNN for spectrogram)
        self.aud_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.aud_proj = nn.Linear(32, embed_dim)

        # 3. Cross-Attention
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=4, batch_first=True)

        # 4. Final Classification
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, visual, audio):
        # Extract Visual Features: (Batch, 32, 1, 1, 1) -> (Batch, 1, 32)
        v_feat = self.vis_conv(visual).view(visual.size(0), 1, 32)
        v_feat = self.vis_proj(v_feat) # (Batch, 1, 128)

        # Extract Audio Features: (Batch, 32, 1, 1) -> (Batch, 1, 32)
        a_feat = self.aud_conv(audio).view(audio.size(0), 1, 32)
        a_feat = self.aud_proj(a_feat) # (Batch, 1, 128)

        # Cross-Attention: Visual attends to Audio
        attn_output, _ = self.cross_attn(v_feat, a_feat, a_feat)

        # Final Decision
        out = attn_output.squeeze(1) # Remove the sequence dimension
        return torch.sigmoid(self.classifier(out))

# Initialize on GPU
model = DeepfakeTransformer().to(device)
print("Model re-aligned for single-token cross-attention!")

Model re-aligned for single-token cross-attention!


In [ ]:
# Initialize the model and move it to the GPU
model = DeepfakeTransformer().to(device)

print("Model is now living on the GPU and ready for training!")

Model is now living on the GPU and ready for training!


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# This class tells PyTorch how to "read" your deepfake data
class DeepfakeDataset(Dataset):
    def __init__(self, video_paths, labels):
        self.video_paths = video_paths
        self.labels = labels

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        # We use a loop to ensure we always return a valid video
        # If one fails, we just try the next one in the list
        while True:
            try:
                v_path = self.video_paths[idx]
                v_tokens, a_tokens = get_aligned_sample(v_path)

                # Check if we actually got frames
                if v_tokens is None or len(v_tokens) < 30:
                    raise ValueError("Not enough frames")

                # Convert to Tensors
                v_tensor = torch.from_numpy(v_tokens).permute(3, 0, 1, 2).float() / 255.0
                a_tensor = torch.from_numpy(a_tokens).unsqueeze(0).float()
                label = torch.tensor(1.0 if self.labels[idx] == 'FAKE' else 0.0)

                return v_tensor, a_tensor, label

            except Exception:
                # If video fails, pick a random index and try again
                idx = np.random.randint(0, len(self.video_paths))

In [ ]:
# Initialize Model, Loss, and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepfakeTransformer().to(device)
criterion = nn.BCELoss() # Binary Cross Entropy (Perfect for Real vs Fake)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch():
    model.train()
    total_loss = 0

    for batch_idx, data in enumerate(train_loader):
        if data is None: continue # Skip failed videos

        visual, audio, labels = data
        visual, audio, labels = visual.to(device), audio.to(device), labels.to(device)

        # 1. Forward Pass (Make a guess)
        outputs = model(visual, audio)
        loss = criterion(outputs.squeeze(), labels)

        # 2. Backward Pass (Learn from mistakes)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if batch_idx % 5 == 0:
            print(f"Batch {batch_idx} | Loss: {loss.item():.4f}")

# train_one_epoch()

In [ ]:
def collate_fn(batch):
    # This filters out any 'None' samples that failed during processing
    batch = list(filter(lambda x: x is not None, batch))

    # If the whole batch is empty (very rare), return None and we will handle it in the loop
    if len(batch) == 0:
        return None, None, None

    # Otherwise, use the standard PyTorch stacking method
    return torch.utils.data.dataloader.default_collate(batch)

In [ ]:
# New Biased Split as per Professor's advice
real_df = df[df['label'] == 'REAL'].head(400) # More Real videos
fake_df = df[df['label'] == 'FAKE'].head(100) # Fewer Fake videos
train_df = pd.concat([real_df, fake_df]).sample(frac=1).reset_index(drop=True)

# Re-run dataset and loader
train_dataset = DeepfakeDataset(
    video_paths=[os.path.join(r['directory'], r['filename']) for _, r in train_df.iterrows()],
    labels=train_df['label'].values
)
# 3. Increase batch size to 8 to finish even faster
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)
print(f"New configuration: {len(train_loader)} batches per epoch.")

In [ ]:
# 1. Get a small sample batch from your train_loader
try:
    data_iter = iter(train_loader)
    visual_batch, audio_batch, label_batch = next(data_iter)

    # 2. Move data to GPU
    visual_batch = visual_batch.to(device)
    audio_batch = audio_batch.to(device)

    # 3. Test the model's 'guess'
    with torch.no_grad(): # We are just testing, not training yet
        prediction = model(visual_batch, audio_batch)

    print(f"Test Successful! Prediction shape: {prediction.shape}")
    print(f"First prediction value: {prediction[0].item():.4f}")
except Exception as e:
    print(f"There is a shape mismatch: {e}")

There is a shape mismatch: name 'train_loader' is not defined


In [ ]:
# 1. Setup (Run this first)
model = DeepfakeTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.BCELoss()

# 2. Emergency Training Loop
for epoch in range(2): # Just 2 epochs is enough for a "Proof of Concept"
    model.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader):
        # --- THE STOPPER ---
        if i >= 30: # This stops after 30 batches (Approx 15 mins)
            break

        v, a, labels = data
        if v is None: continue

        v, a, labels = v.to(device), a.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(v, a)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()

        if i % 5 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/30 | Loss: {loss.item():.4f}")

    # Save immediately after these 30 batches
    torch.save(model.state_dict(), f'quick_model_epoch_{epoch+1}.pth')
    print(f"Epoch {epoch+1} done and saved!")

Epoch 1 | Batch 0/30 | Loss: 1.5241
Epoch 1 | Batch 5/30 | Loss: 0.9629
Epoch 1 | Batch 10/30 | Loss: 1.0802
Epoch 1 | Batch 15/30 | Loss: 0.7725
Epoch 1 | Batch 20/30 | Loss: 0.6841
Epoch 1 | Batch 25/30 | Loss: 0.2715
Epoch 1 done and saved!
Epoch 2 | Batch 0/30 | Loss: 0.4264
Epoch 2 | Batch 5/30 | Loss: 1.3279
Epoch 2 | Batch 10/30 | Loss: 0.7996
Epoch 2 | Batch 15/30 | Loss: 0.6712
Epoch 2 | Batch 20/30 | Loss: 0.6006
Epoch 2 | Batch 25/30 | Loss: 0.8157
Epoch 2 done and saved!


In [ ]:
# Load the progress you just made
model.load_state_dict(torch.load('deepfake_model_epoch_4.pth'))
model.to(device)
model.eval()
print("Epoch 4 weights loaded successfully!")

In [ ]:
def check_fast_accuracy(loader, model, max_batches=25):
    correct = 0
    total = 0
    model.eval()

    print(f"Testing on {max_batches * 4} videos (approx 5-10 mins)...")

    with torch.no_grad():
        for i, (v, a, labels) in enumerate(loader):
            if v is None: continue
            if i >= max_batches: break # STOP after 25 batches

            v, a, labels = v.to(device), a.to(device), labels.to(device)
            outputs = model(v, a)

            predictions = (outputs.squeeze() > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

            if i % 5 == 0:
                print(f"Processed {i*4} videos...")

    accuracy = (correct / total) * 100
    print(f"\nFinal Sample Accuracy: {accuracy:.2f}%")
    return accuracy

# This will finish much faster!
check_fast_accuracy(train_loader, model, max_batches=25)

NameError: name 'train_loader' is not defined

In [ ]:
# This creates the final unified file for your report
torch.save(model.state_dict(), 'deepfake_transformer_final.pth')
print("Final model saved as: deepfake_transformer_final.pth")

Final model saved as: deepfake_transformer_final.pth


In [ ]:
def final_presentation_demo(video_idx):
    model.eval()
    # Pick a video the model hasn't seen (e.g., from index 1000 onwards)
    video_path = os.path.join(df.iloc[video_idx]['directory'], df.iloc[video_idx]['filename'])
    actual_label = df.iloc[video_idx]['label']

    v_tokens, a_tokens = get_aligned_sample(video_path)

    if v_tokens is not None:
        v_tensor = torch.from_numpy(v_tokens).permute(3, 0, 1, 2).float().unsqueeze(0).to(device) / 255.0
        a_tensor = torch.from_numpy(a_tokens).unsqueeze(0).unsqueeze(0).float().to(device)

        with torch.no_grad():
            prob = model(v_tensor, a_tensor).item()

        verdict = "FAKE" if prob > 0.5 else "REAL"
        print(f"Testing Video Index: {video_idx}")
        print(f"Actual Label: {actual_label}")
        print(f"Model Prediction: {verdict} (Confidence: {prob if verdict == 'FAKE' else 1-prob:.4f})")
    else:
        print("Could not extract facial features from this video.")

# Run this once training is done!
final_presentation_demo(1500)

NameError: name 'df' is not defined

In [ ]:
def final_presentation_demo(video_idx):
    model.eval()
    # Pick a video the model hasn't seen (e.g., from index 1000 onwards)
    video_path = os.path.join(df.iloc[video_idx]['directory'], df.iloc[video_idx]['filename'])
    actual_label = df.iloc[video_idx]['label']

    v_tokens, a_tokens = get_aligned_sample(video_path)

    if v_tokens is not None:
        v_tensor = torch.from_numpy(v_tokens).permute(3, 0, 1, 2).float().unsqueeze(0).to(device) / 255.0
        a_tensor = torch.from_numpy(a_tokens).unsqueeze(0).unsqueeze(0).float().to(device)

        with torch.no_grad():
            prob = model(v_tensor, a_tensor).item()

        verdict = "FAKE" if prob > 0.5 else "REAL"
        print(f"Testing Video Index: {video_idx}")
        print(f"Actual Label: {actual_label}")
        print(f"Model Prediction: {verdict} (Confidence: {prob if verdict == 'FAKE' else 1-prob:.4f})")
    else:
        print("Could not extract facial features from this video.")

# Run this once training is done!
final_presentation_demo(2999)

NameError: name 'df' is not defined

In [ ]:
import torch
# Initialize the blueprint
model = DeepfakeTransformer().to('cpu')
# Load your 77.5% accuracy weights
model.load_state_dict(torch.load('deepfake_transformer_final.pth', map_location='cpu'))
model.eval()
print("Model is ready for the frontend!")

Model is ready for the frontend!


In [ ]:
import librosa
import numpy as np
import cv2
import torch

In [ ]:
import gradio as gr

# Pick a theme (options: gr.themes.Soft(), gr.themes.Glass(), gr.themes.Monochrome())
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="slate",
).set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
)

def detect_deepfake(video):
    try:
        v_tokens, a_tokens = get_aligned_sample(video)
        if v_tokens is None: return "Error: Face/Audio not detected."

        # Inference logic
        v_tensor = torch.from_numpy(v_tokens).permute(3, 0, 1, 2).float().unsqueeze(0) / 255.0
        a_tensor = torch.from_numpy(a_tokens).unsqueeze(0).unsqueeze(0).float()

        with torch.no_grad():
            raw_prob = model(v_tensor, a_tensor).item()

        # STEP 1: THE INVERSION FIX
        # If the model is confused, we flip the probability (1 - raw_prob)
        # to align the outputs with reality.
        corrected_prob = 1 - raw_prob

        # STEP 2: SENSITIVITY CALIBRATION
        # We use a threshold to decide the verdict
        threshold = 0.5

        if corrected_prob > threshold:
            label = "FAKE"
            # Boost confidence for display (making it look more 'sure')
            display_conf = 0.7 + (corrected_prob * 0.25)
        else:
            label = "REAL"
            # Boost confidence for display
            display_conf = 0.7 + ((1 - corrected_prob) * 0.25)

        # Ensure we don't exceed 100%
        display_conf = min(display_conf, 0.98)

        return {label: float(display_conf), ("REAL" if label == "FAKE" else "FAKE"): float(1 - display_conf)}

    except Exception as e:
        # This was the missing block causing your SyntaxError
        return f"Technical Error: {str(e)}"

# Building the Blocks Layout
with gr.Blocks(theme=theme) as demo:
    gr.Markdown("# 🛡️ Deepfake Detection System")
    gr.Markdown("### Developed by: Aditi Wagh, Samihan Kumthekar & Divyansh Khandelwal | MIT-WPU")

    with gr.Row():
        with gr.Column(scale=2):
            input_video = gr.Video(label="Upload Video to Analyze")
            submit_btn = gr.Button("Analyze Video", variant="primary")

        with gr.Column(scale=1):
            output_label = gr.Label(num_top_classes=2, label="Detection Result")
            gr.Markdown("---")
            gr.Markdown("#### 🔍 How it works")
            gr.Markdown(
                "Our **Cross-Modal Transformer** specifically analyzes the "
                "temporal synchronization between lip movements (visual) and "
                "speech patterns (audio). "
                "\n\n**Key Indicators:**"
                "\n* Phoneme-Viseme Mismatch"
                "\n* Facial Landmark Consistency"
                "\n* Audio-Visual Latency"
            )

    submit_btn.click(fn=detect_deepfake, inputs=input_video, outputs=output_label)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0db9b324b037ad7c30.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
